In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_23312\4208644247.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'I do not have access to a specific, private database, and you have not specified which database you are referring to.\n\nThere are many different music and art databases, each with a different number of artists. For example:\n\n*   **MusicBrainz:** A community-maintained open database. Its size changes daily as users add new data.\n*   **Discogs:** A massive database of music releases and artists.\n*   **Encyclopaedia Metallum:** Specifically focuses on metal music, with hundreds of thousands of artists.\n*   **AI Training Datasets:** Datasets used to train AI models (like the one mentioned in the Midjourney/Stability AI lawsuit) often contain lists of thousands or millions of artists scraped from various sources.\n\n**To give you an accurate answer, please clarify which database you are asking about.**', 'extras': {'signature': 'EnMKcQFpFH0TVytNbPG+E3/0/16i9OsYObDbT1fNN/bmtRjbC06u+7vu12HRgy5PHR0UTleoMcliRjJ7z5g4yHYLPyk9Pre73bD2BV8mha06HyJYUJMswsKw9IEMKpJ3dX6